## **Agents using Langchain**

In [46]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq

from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

import requests

In [47]:
search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke("Top news in India")

In [48]:
print(results)

The top 10 cryptocurrency news media in India encompass leading digital platforms dedicated to delivering specialized coverage of blockchain technologies, cryptocurrency market analyses, regulatory shifts, and India-specific developments, particular… Hindustan Times: Stay updated with Hindustan Times for the top India news, world events, and breaking stories. Get real-time updates on politics, wars, live cricket scores, entertainment... 36 minutes ago · Today's top India news headlines, news on Indian politics, elections, government, business, technology, and Bollywood. India News Today: Get latest India news and breaking news from India on Times of India. TOI brings the top news headlines from India on Politics, Elections, Current Affairs and more. 1 hour ago · Get the latest India news, breaking headlines, political developments, government updates, weather alerts, crime news and state coverage at CNBC TV18. Stay here for real-time updates on breaking news from India and across the w

In [ ]:
@tool
def get_weather_data(city: str) -> str:
    """
        This function fetches the current weather data for a given city
    """

    url = f"http://api.weatherstack.com/current?access_key={YOUR_API_KEY}={city}"
    response = requests.get(url)

    return response.json()


In [50]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [51]:
llm.invoke("Hi")

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EvgFCvUFAWkUfRNjcfzHFNY04dW6bNJLInIo8KwNVvUL8XLY7RXGdMBJdSaBI36hmjuDNJyR9BSdaAwn3sCsJZVVRxxZnN8pNWVEioM3JVKUqXS8tg2zLqjdBu2Ko2+7flXgfPh1HbUlmUNguY1o+iq+OFAIh87ktWCZpNxLOPsurTYVZHKDsF3n48Hz+JjRAsDEEOr66GgxQTscntbIO5N7HE/6Iu8wOFxf9aqBkotNSBhAy4Clk2HBSYeHOrbLl61gCCJ/cam237iO3ixs6ZOES9g7yULL8zM36bGHvyCC+InbZNr5BkEimN1lJOx5FekQdk+sKE6555ba+GYUiLEQwGAzr2ASJFpJF14uqdG0HPkaghhd0cW6MeXRMzUesAbl1ArmEqxbiBcfU5DUdTZf+4zEp334uS/G4XLgvmSK6gF8MGreTCHy6WEtXG2b9oIqs+VpuRJz0/Z83wJW9zWaspRE/F1dIdCyx+kwP0X8RPYhKMpAJ1vW4pK3JGXWKJzlrop68QYk28djiP9ImpIjVnyUOSl2KPUMFSlgpp1P5wKjs6olfnu/aOaWkKNVOCQpyTzWshICQ/GkbnHzliReBd1Q6cnuvH7vDpZwKpxS5jm5hJ74CBjcujoCD4QHRuebG1EPl1lLRAwLruG3EO1+9AQZXOwH8VHyRSuHrDCTEnVfkRByDziwkbAfGMqMmyVxHbM2V+RO0f3wPQIxU4IpPgWTCkhYER0kKJ55gRyx8AlBitV5R4i8zS3WmqJcVpxf/J3rFkD2Zi87oxGQP/HCZKXAD8YiCD229+uDGnLvDP2WTbQawTjez99c/3CrElA9+loWGfJZ6KW/h9R9rR9bZg5sZgeE0kBtINJM6MK5JQCdi5v0rEdcpcq8+N0

In [52]:
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic import hub

In [53]:
## step 2: pull the react prompt inside hub

from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True
)

print(prompt)

input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


In [54]:
## step 3: Create the react agent manually with the pulled prompt

agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

In [55]:
## step 4: Wrap it with Agent Executor

agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True
)

In [56]:
response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})

print(response)



> Entering new AgentExecutor chain...
Action: get_weather_data
Action Input: Bhopal{'request': {'type': 'City', 'query': 'Bhopal, India', 'language': 'en', 'unit': 'm'}, 'location': {'name': 'Bhopal', 'country': 'India', 'region': 'Madhya Pradesh', 'lat': '23.267', 'lon': '77.400', 'timezone_id': 'Asia/Kolkata', 'localtime': '2026-11-16 20:04', 'localtime_epoch': 1794859440, 'utc_offset': '5.50'}, 'current': {'observation_time': '02:34 PM', 'temperature': 22, 'weather_code': 149, 'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0006_mist.png'], 'weather_descriptions': ['Smoky haze'], 'astro': {'sunrise': '06:34 AM', 'sunset': '05:36 PM', 'moonrise': '12:06 PM', 'moonset': '11:22 PM', 'moon_phase': 'Waxing Crescent', 'moon_illumination': 36}, 'air_quality': {'co': '418', 'no2': '16.6', 'o3': '76', 'so2': '9', 'pm2_5': '46.4', 'pm10': '50.6', 'us-epa-index': '3', 'gb-defra-index': '3'}, 'wind_speed': 4, 'wind_degree': 36, 'wind_dir': 'NE', 'pressur

In [58]:
print(response['output'])

The capital of Madhya Pradesh is **Bhopal**. 

Current weather conditions in Bhopal:
- **Temperature:** 22°C (feels like 22°C)
- **Condition:** Smoky haze
- **Humidity:** 53%
- **Wind:** 4 km/h from NE
- **Cloud Cover:** 6%
